## PyTorch ka Introduction

> Yeh notebook [AI for Beginners Curricula](http://github.com/microsoft/ai-for-beginners) ka hissa hai. Poori learning material ke liye repository par jaayein.

### Neural Frameworks

Humne seekha hai ki neural network ko train karne ke liye aapko chahiye:
* Matrices (tensors) ko tezi se multiply karna
* Gradients calculate karna taaki gradient descent optimization ho sake

Neural network frameworks aapko yeh karne dete hain:
* Tensors ke saath us compute par kaam karna jo available ho, chahe woh CPU ho, GPU ho, ya TPU bhi ho
* Gradients ko automatically calculate karna (yeh sabhi built-in tensor functions ke liye explicitly programmed hote hain)

Optionally:
* Neural network constructor / high-level API (network ko layers ke sequence ke roop mein describe karna)
* Simple training functions (`fit`, jaise Scikit Learn mein hai)
* Gradient descent ke alawa kai optimization algorithms
* Data handling abstractions (jo ideally GPU par bhi kaam karein)

### Sabse popular frameworks

* Tensorflow 1.x - pehla widely available framework (Google). Isme aap static computational graph define karte the, use GPU par bhejte the aur explicitly evaluate karte the.
* PyTorch - Facebook ka framework jo popularity mein badh raha hai.
* Keras - Tensorflow/PyTorch ke upar high-level API, jo neural networks ka use unified aur simple banata hai (Francois Chollet).
* Tensorflow 2.x + Keras - Tensorflow ka naya version jisme Keras ki functionality integrated hai, aur jo **dynamic computational graph** support karta hai, jisse numpy (aur PyTorch) jaise tensor operations karna possible hota hai.

Is Notebook mein hum PyTorch use karna seekhenge. Aapko yeh ensure karna hoga ki aapke paas PyTorch ka recent version installed ho - iske liye [unki site par diye gaye instructions](https://pytorch.org/get-started/locally/) follow karein. Yeh aam taur par itna simple hota hai:
```
pip install torch torchvision
```
ya
```
conda install pytorch -c pytorch
```

In [ ]:
import torch
torch.__version__

## Basic concepts: Tensor

**Tensor** ek multi-dimensional array hai. Alag-alag types ke data ko represent karne ke liye tensors use karna bahut convenient hai:
* 400x400 - black-and-white picture
* 400x400x3 - color picture
* 16x400x400x3 - 16 color pictures ka minibatch
* 25x400x400x3 - 25 frames per second wala ek second ka video
* 8x25x400x400x3 - 8 ek-second ke videos ka minibatch

### Simple tensors

Aap np-arrays ki list se aasaani se simple tensors bana sakte hain, ya random tensors generate kar sakte hain:

In [ ]:
a = torch.tensor([[1,2],[3,4]])
print(a)
a = torch.randn(size=(10,3))
print(a)

Aap tensors par arithmetic operations kar sakte hain, jo element-wise hote hain, bilkul numpy ki tarah. Zaroorat padne par tensors automatically required dimension tak expand ho jaate hain. Tensor se numpy-array nikalne ke liye `.numpy()` use karein:

In [ ]:
print(a-a[0])
print(torch.exp(a)[0].numpy())

## In-place aur out-of-place operations

`+`/`add` jaise tensor operations naye tensors return karte hain. Lekin kabhi-kabhi aapko existing tensor ko in-place modify karna padta hai. Zyadatar operations ke in-place equivalents hote hain, jo `_` par khatam hote hain:

In [ ]:
u = torch.tensor(5)
print("Result when adding out-of-place:",u.add(torch.tensor(3)))
u.add_(torch.tensor(3))
print("Result after adding in-place:", u)

Yeh dekhiye, hum matrix ki saari rows ka sum ek simple tareeke se kaise nikal sakte hain:

In [ ]:
s = torch.zeros_like(a[0])
for i in a:
  s.add_(i)

print(s)

In [ ]:
torch.sum(a,axis=0)

PyTorch tensors ke baare mein aap [official documentation](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html) mein aur padh sakte hain.

## Gradients compute karna

Back propagation ke liye aapko gradients compute karne padte hain. Hum kisi bhi PyTorch Tensor ka `requires_grad` attribute `True` set kar sakte hain, jisse is tensor ke saath kiye gaye saare operations gradient calculation ke liye track honge. Gradients compute karne ke liye aapko `backward()` method call karna hoga, jiske baad gradient `grad` attribute ke zariye available ho jayega:

In [ ]:
a = torch.randn(size=(2, 2), requires_grad=True)
b = torch.randn(size=(2, 2))

c = torch.mean(torch.sqrt(torch.square(a) + torch.square(b)))  # Do some math using `a`
c.backward() # call backward() to compute all gradients
# What's the gradient of `c` with respect to `a`?
print(a.grad)

Zyada precise baat karein toh, PyTorch **gradients** ko automatically accumulate karta hai. Agar aap `backward` call karte waqt `retain_graph=True` specify karte hain, toh computational graph preserved rehta hai, aur naya gradient `grad` field mein add ho jata hai. Gradients ki calculation ko shuru se start karne ke liye, hamein `zero_()` call karke `grad` field ko explicitly 0 par reset karna hoga:

In [ ]:
c = torch.mean(torch.sqrt(torch.square(a) + torch.square(b)))
c.backward(retain_graph=True)
c.backward(retain_graph=True)
print(a.grad)
a.grad.zero_()
c.backward()
print(a.grad)

Gradients compute karne ke liye PyTorch **compute graph** banata aur maintain karta hai. Har us tensor ke liye jiska `requires_grad` flag `True` set hota hai, PyTorch ek special function `grad_fn` maintain karta hai, jo chain differentiation rule ke hisaab se expression ka derivative calculate karta hai:

In [ ]:
print(c)

Yahan `c` ko `mean` function se compute kiya gaya hai, isliye `grad_fn` ek function ki taraf point karta hai jise `MeanBackward` kehte hain.

Zyadatar cases mein hum chahte hain ki PyTorch kisi scalar function (jaise loss function) ka gradient calculate kare. Lekin agar hum kisi tensor ka gradient kisi doosre tensor ke respect mein calculate karna chahte hain, toh PyTorch hamein Jacobian matrix aur diye gaye vector ka product compute karne deta hai.

Maan lijiye hamare paas ek vector function $\vec{y}=f(\vec{x})$ hai, jahan
$\vec{x}=\langle x_1,\dots,x_n\rangle$ aur
$\vec{y}=\langle y_1,\dots,y_m\rangle$, toh $\vec{x}$ ke respect mein $\vec{y}$ ka gradient **Jacobian** se define hota hai:

$$
\begin{align}J=\left(\begin{array}{ccc}
   \frac{\partial y_{1}}{\partial x_{1}} & \cdots & \frac{\partial y_{1}}{\partial x_{n}}\\
   \vdots & \ddots & \vdots\\
   \frac{\partial y_{m}}{\partial x_{1}} & \cdots & \frac{\partial y_{m}}{\partial x_{n}}
\end{array}\right)\end{align}
$$

PyTorch hamein poora Jacobian dene ke bajaye, Jacobian aur kisi vector $v=(v_1 \dots v_m)$ ka product $v^T\cdot J$ compute karta hai. Iske liye hamein ``backward`` call karna hota hai aur `v` ko argument ke roop mein pass karna hota hai. `v` ka size wahi hona chahiye jo us tensor ka size hai jiske respect mein hum gradient compute kar rahe hain.

In [ ]:
c = torch.sqrt(torch.square(a) + torch.square(b))
c.backward(torch.eye(2)) # eye(2) means 2x2 identity matrix
print(a.grad)

# Example 0: Gradient Descent se optimization

Chaliye automatic differentiation use karke ek simple two-variable function $f(x_1,x_2)=(x_1-3)^2+(x_2+2)^2$ ka minimum dhundhne ki koshish karte hain. Value `x` current point ke coordinates ko store karti hai. Hum initial point $x^{(0)}=(0,0)$ se shuru karte hain aur gradient descent formula use karke sequence ka agla point calculate karte hain:
$$
x^{(n+1)} = x^{(n)} - \eta\nabla f
$$
Yahan $\eta$ ko **learning rate** kehte hain (code mein ise `lr` se represent kiya jayega), aur $\nabla f = (\frac{\partial f}{\partial x_1},\frac{\partial f}{\partial x_2})$ - $f$ ka gradient.

Shuru karne ke liye, `x` ki initial value aur function `f` define karte hain:

In [ ]:
x = torch.zeros(2,requires_grad=True)
f = lambda x : (x-torch.tensor([3,-2])).pow(2).sum()
lr = 0.1

Ab gradient descent ki 15 iterations karte hain. Har iteration mein hum `x` coordinates ko update karenge aur print karenge, taaki yeh confirm ho sake ki hum minimum point (3, -2) ki taraf badh rahe hain:

In [ ]:
for i in range(15):
    y = f(x)
    y.backward()
    gr = x.grad
    x.data.add_(-lr*gr)
    x.grad.zero_()
    print("Step {}: x[0]={}, x[1]={}".format(i,x[0],x[1]))

## Example 1: Linear Regression

Ab hum **linear regression** ki classical problem solve karne ke liye kaafi jaante hain. Chaliye ek chhota synthetic dataset banate hain:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
import random

In [ ]:
np.random.seed(13) # pick the seed for reproducibility - change it to explore the effects of random variations

train_x = np.linspace(0, 3, 120)
train_labels = 2 * train_x + 0.9 + np.random.randn(*train_x.shape) * 0.5

plt.scatter(train_x,train_labels)

Linear regression ek straight line $f_{W,b}(x) = Wx+b$ se define hota hai, jahan $W, b$ model parameters hain jo hamein dhundhne hain. Hamare dataset $\{x_i,y_u\}_{i=1}^N$ par error (jise **loss function** bhi kehte hain) ko mean square error ke roop mein define kiya ja sakta hai:
$$
\mathcal{L}(W,b) = {1\over N}\sum_{i=1}^N (f_{W,b}(x_i)-y_i)^2
$$

Chaliye apna model aur loss function define karte hain:

In [ ]:
input_dim = 1
output_dim = 1
learning_rate = 0.1

# This is our weight matrix
w = torch.tensor([100.0],requires_grad=True,dtype=torch.float32)
# This is our bias vector
b = torch.zeros(size=(output_dim,),requires_grad=True)

def f(x):
  return torch.matmul(x,w) + b

def compute_loss(labels, predictions):
  return torch.mean(torch.square(labels - predictions))

Hum model ko minibatches ki series par train karenge. Hum gradient descent use karenge, aur neeche diye formulas se model ke parameters adjust karenge:
$$
\begin{array}{l}
W^{(n+1)}=W^{(n)}-\eta\frac{\partial\mathcal{L}}{\partial W} \\
b^{(n+1)}=b^{(n)}-\eta\frac{\partial\mathcal{L}}{\partial b} \\
\end{array}
$$

In [ ]:
def train_on_batch(x, y):
  predictions = f(x)
  loss = compute_loss(y, predictions)
  loss.backward()
  w.data.sub_(learning_rate * w.grad)
  b.data.sub_(learning_rate * b.grad)
  w.grad.zero_()
  b.grad.zero_()
  return loss

Chalo training karte hain. Hum dataset ke through kai baar (jise **epochs** kehte hain) guzrenge, use minibatches mein divide karenge aur upar define kiye gaye function ko call karenge:

In [ ]:
# Shuffle the data.
indices = np.random.permutation(len(train_x))
features = torch.tensor(train_x[indices],dtype=torch.float32)
labels = torch.tensor(train_labels[indices],dtype=torch.float32)

In [ ]:
batch_size = 4
for epoch in range(10):
  for i in range(0,len(features),batch_size):
    loss = train_on_batch(features[i:i+batch_size].view(-1,1),labels[i:i+batch_size])
  print('Epoch %d: last batch loss = %.4f' % (epoch, float(loss)))

Ab humein optimized parameters $W$ aur $b$ mil gaye hain. Dhyan dein ki unki values lagbhag wahi hain jo dataset banate waqt use ki gayi thi ($W=2, b=1$).

In [ ]:
w,b

In [ ]:
plt.scatter(train_x,train_labels)
x = np.array([min(train_x),max(train_x)])
with torch.no_grad():
  y = w.numpy()*x+b.numpy()
plt.plot(x,y,color='red')

## GPU par computations

Computations ke liye GPU use karne ke liye, PyTorch tensors ko GPU par move karne aur GPU ke liye computational graph banane ko support karta hai. Traditionally, code ki shuruaat mein hum available compute device `device` (jo ya toh `cpu` hota hai ya `cuda`) define karte hain, aur phir `.to(device)` call se saare tensors ko is device par move karte hain. Hum tensors ko seedhe specified device par bhi bana sakte hain, tensor creation code mein `device=...` parameter pass karke. Aisa code bina kisi change ke CPU aur GPU dono par kaam karta hai:

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Doing computations on '+device)

### Changes here: indicate device
w = torch.tensor([100.0],requires_grad=True,dtype=torch.float32,device=device)
b = torch.zeros(size=(output_dim,),requires_grad=True,device=device)

def f(x):
  return torch.matmul(x,w) + b

def compute_loss(labels, predictions):
  return torch.mean(torch.square(labels - predictions))

def train_on_batch(x, y):
  predictions = f(x)
  loss = compute_loss(y, predictions)
  loss.backward()
  w.data.sub_(learning_rate * w.grad)
  b.data.sub_(learning_rate * b.grad)
  w.grad.zero_()
  b.grad.zero_()
  return loss

batch_size = 4
for epoch in range(10):
  for i in range(0,len(features),batch_size):
    ### Changes here: move data to required device
    loss = train_on_batch(features[i:i+batch_size].view(-1,1).to(device),labels[i:i+batch_size].to(device))
  print('Epoch %d: last batch loss = %.4f' % (epoch, float(loss)))

## Example 2: Classification

Ab hum binary classification problem dekhenge. Aisi problem ka ek achha example hai tumor ko classify karna, jisme uske size aur age ke basis par decide karna hai ki woh malignant hai ya benign.

Main model regression jaisa hi hai, lekin hamein alag loss function use karna hoga. Chaliye sample data taiyaar karke shuru karte hain:

In [ ]:
np.random.seed(0) # pick the seed for reproducibility - change it to explore the effects of random variations

n = 100
X, Y = make_classification(n_samples = n, n_features=2,
                           n_redundant=0, n_informative=2, flip_y=0.1,class_sep=1.5)
X = X.astype(np.float32)
Y = Y.astype(np.int32)

split = [ 70*n//100, (15+70)*n//100 ]
train_x, valid_x, test_x = np.split(X, split)
train_labels, valid_labels, test_labels = np.split(Y, split)

In [ ]:
def plot_dataset(features, labels, W=None, b=None):
    # prepare the plot
    fig, ax = plt.subplots(1, 1)
    ax.set_xlabel('$x_i[0]$ -- (feature 1)')
    ax.set_ylabel('$x_i[1]$ -- (feature 2)')
    colors = ['r' if l else 'b' for l in labels]
    ax.scatter(features[:, 0], features[:, 1], marker='o', c=colors, s=100, alpha = 0.5)
    if W is not None:
        min_x = min(features[:,0])
        max_x = max(features[:,1])
        min_y = min(features[:,1])*(1-.1)
        max_y = max(features[:,1])*(1+.1)
        cx = np.array([min_x,max_x],dtype=np.float32)
        cy = (0.5-W[0]*cx-b)/W[1]
        ax.plot(cx,cy,'g')
        ax.set_ylim(min_y,max_y)
    fig.show()

In [ ]:
plot_dataset(train_x, train_labels)

## Single-layer Perceptron ko train karna

Chaliye PyTorch ke gradient computation mechanism se single-layer perceptron ko train karte hain.

Hamare neural network mein 2 inputs aur 1 output hoga. Weight matrix $W$ ka size $2\times1$ hoga, aur bias vector $b$ -- $1$.

Apne code ko zyada structured banane ke liye, chaliye saare parameters ko ek hi class mein group karte hain:

In [ ]:
class Network():
  def __init__(self):
     self.W = torch.randn(size=(2,1),requires_grad=True)
     self.b = torch.zeros(size=(1,),requires_grad=True)

  def forward(self,x):
    return torch.matmul(x,self.W)+self.b

  def zero_grad(self):
    self.W.data.zero_()
    self.b.data.zero_()

  def update(self,lr=0.1):
    self.W.data.sub_(lr*self.W.grad)
    self.b.data.sub_(lr*self.b)

net = Network()

> Dhyan dein ki hum `W.zero_()` ke bajaye `W.data.zero_()` use karte hain. Aisa isliye karna padta hai kyunki hum us tensor ko seedhe modify nahi kar sakte jo *Autograd* mechanism se track ho raha ho.

Main model pehle wale example jaisa hi rahega, lekin loss function logistic loss hoga. Logistic loss apply karne ke liye, hamein apne network ke output ke roop mein **probability** ki value chahiye, yaani output $z$ ko [0,1] range mein laane ke liye hamein `sigmoid` activation function use karna hoga: $p=\sigma(z)$.

Agar hamein i-th input value ke liye actual class $y_i\in\{0,1\}$ ke corresponding probability $p_i$ milti hai, toh hum loss ko is tarah calculate karte hain: $\mathcal{L_i}=-(y_i\log p_i + (1-y_i)log(1-p_i))$.

PyTorch mein, yeh dono steps (sigmoid apply karna aur phir logistic loss) `binary_cross_entropy_with_logits` function ke ek hi call se ho sakte hain. Kyunki hum apne network ko minibatches mein train kar rahe hain, hamein minibatch ke saare elements par loss ka average lena hoga - aur yeh bhi `binary_cross_entropy_with_logits` function automatically kar deta hai:

> `binary_crossentropy_with_logits` call karna, `sigmoid` call karne ke baad `binary_crossentropy` call karne ke barabar hai.

In [ ]:
def train_on_batch(net, x, y):
  z = net.forward(x).flatten()
  loss = torch.nn.functional.binary_cross_entropy_with_logits(input=z,target=y)
  net.zero_grad()
  loss.backward()
  net.update()
  return loss

Apne data par loop karne ke liye hum PyTorch ka built-in mechanism use karenge jo datasets manage karne ke liye bana hai. Yeh do main concepts par based hai:
* **Dataset** data ka main source hai, yeh ya toh **Iterable** ho sakta hai ya **Map-style**.
* **Dataloader** dataset se data load karne aur use minibatches mein divide karne ke liye responsible hota hai.

Hamare case mein, hum ek tensor par based dataset define karenge aur use 16 elements ke minibatches mein divide karenge. Har minibatch mein do tensors hote hain: input data (size=16x2) aur labels (integer type ka 16 length wala ek vector - class numbers).

In [ ]:
# Create a tf.data.Dataset object for easy batched iteration
dataset = torch.utils.data.TensorDataset(torch.tensor(train_x),torch.tensor(train_labels,dtype=torch.float32))
dataloader = torch.utils.data.DataLoader(dataset,batch_size=16)

list(dataloader)[0]

In [ ]:
for epoch in range(15):
  for (x, y) in dataloader:
    loss = train_on_batch(net,x,y)
  print('Epoch %d: last batch loss = %.4f' % (epoch, float(loss)))

In [ ]:
print(net.W,net.b)

In [ ]:
plot_dataset(train_x,train_labels,net.W.detach().numpy(),net.b.detach().numpy())

In [ ]:
pred = torch.sigmoid(net.forward(torch.tensor(valid_x)))
torch.mean(((pred.view(-1)>0.5)==(torch.tensor(valid_labels)>0.5)).type(torch.float32))

Chaliye samajhte hain ki yahan kya ho raha hai:
* `pred` poore validation dataset ke liye predicted probabilities ka ek vector hai. Ise hum original validation data `valid_x` ko apne network se chalakar aur `sigmoid` apply karke probabilities nikalte hain.
* `pred.view(-1)` original tensor ka ek flattened view banata hai. `view` numpy ke `reshape` function jaisa hai.
* `pred.view(-1)>0.5` ek boolean tensor ya truth values return karta hai jo predicted class dikhata hai (False = class 0, True = class 1).
* Isi tarah, `torch.tensor(valid_labels)>0.5)` validation labels ke liye truth values ka boolean tensor banata hai.
* Hum in dono tensors ki element-wise comparison karte hain aur ek aur boolean tensor paate hain, jahan `True` sahi prediction ko dikhata hai aur `False` galat ko.
* Hum us tensor ko floating point mein badalte hain aur `torch.mean` se uski mean value lete hain - yahi required accuracy hai.

## Neural Networks aur Optimizers

PyTorch mein neural network ko represent karne ke liye ek special module `torch.nn.Module` define kiya gaya hai. Apna khud ka neural network define karne ke do tareeke hain:
* **Sequential**, jisme aap sirf apne network mein shamil layers ki list specify karte hain
* Ek **class** ke roop mein, jo `torch.nn.Module` se inherit ki gayi ho

Pehla tareeka aapko layers ke sequential combination ke saath standard networks specify karne deta hai, jabki doosra tareeka zyada flexible hai aur aapko kisi bhi complex architecture ka network express karne ka mauka deta hai.

Modules ke andar aap standard **layers** use kar sakte hain, jaise:
* `Linear` - dense linear layer, jo single-layer perceptron ke barabar hai. Iska architecture wahi hai jo humne upar apne network ke liye define kiya tha
* `Softmax`, `Sigmoid`, `ReLU` - layers jo activation functions ke corresponding hain
* Special types ke networks ke liye aur bhi layers hain - jaise convolution, recurrent aadi. Inme se kai ko hum course mein aage phir dekhenge.

> PyTorch mein zyadatar activation functions aur loss functions do forms mein available hain: ek **function** ke roop mein (`torch.nn.functional` namespace ke andar) aur **ek layer** ke roop mein (`torch.nn` namespace ke andar). Activation functions ke liye, aksar `torch.nn.functional` ke functional elements use karna aasaan hota hai, alag se layer object banaye bina.

Agar hum single-layer perceptron ko train karna chahte hain, toh hum bas ek built-in `Linear` layer use kar sakte hain:

In [ ]:
net = torch.nn.Linear(2,1) # 2 inputs, 1 output

print(list(net.parameters()))

Jaisa ki aap dekh sakte hain, `parameters()` method un sabhi parameters ko return karta hai jinhe training ke dauran adjust karna hota hai. Yeh weight matrix $W$ aur bias $b$ se related hote hain. Aap dekh sakte hain ki inka `requires_grad` `True` set hai, kyunki hamein parameters ke respect mein gradients compute karne hote hain.

PyTorch mein built-in **optimizers** bhi hote hain, jo **gradient descent** jaise optimization methods implement karte hain. Yeh dekhiye hum **stochastic gradient descent optimizer** kaise define kar sakte hain:

In [ ]:
optim = torch.optim.SGD(net.parameters(),lr=0.05)

Optimizer use karte hue, hamara training loop kuch aisa dikhega:

In [ ]:
val_x = torch.tensor(valid_x)
val_lab = torch.tensor(valid_labels)

for ep in range(10):
  for (x,y) in dataloader:
    z = net(x).flatten()
    loss = torch.nn.functional.binary_cross_entropy_with_logits(z,y)
    optim.zero_grad()
    loss.backward()
    optim.step()
  acc = ((torch.sigmoid(net(val_x).flatten())>0.5).float()==val_lab).float().mean()
  print(f"Epoch {ep}: last batch loss = {loss}, val acc = {acc}")

Aap dekh sakte hain ki hum apne network ko input data par apply karne ke liye `net.forward(x)` ki jagah `net(x)` use kar sakte hain, kyunki `nn.Module` ne Python ka `__call__()` function implement kiya hai.

Is baat ko dhyan mein rakhte hue, hum ek generic `train` function define kar sakte hain:

In [ ]:
def train(net, dataloader, val_x, val_lab, epochs=10, lr=0.05):
  optim = torch.optim.Adam(net.parameters(),lr=lr)
  for ep in range(epochs):
    for (x,y) in dataloader:
      z = net(x).flatten()
      loss = torch.nn.functional.binary_cross_entropy_with_logits(z,y)
      optim.zero_grad()
      loss.backward()
      optim.step()
    acc = ((torch.sigmoid(net(val_x).flatten())>0.5).float()==val_lab).float().mean()
    print(f"Epoch {ep}: last batch loss = {loss}, val acc = {acc}")

net = torch.nn.Linear(2,1)

train(net,dataloader,val_x,val_lab,lr=0.03)

## Network ko layers ke sequence ke roop mein define karna

Ab chaliye multi-layered perceptron ko train karte hain. Ise bas layers ka sequence specify karke define kiya ja sakta hai. Resulting object automatically `Module` se inherit karega, yaani usme `parameters` method bhi hoga, jo poore network ke saare parameters return karega.

In [ ]:
net = torch.nn.Sequential(torch.nn.Linear(2,5),torch.nn.Sigmoid(),torch.nn.Linear(5,1))
print(net)

Hum is multi-layer network ko upar define kiye gaye `train` function se train kar sakte hain:

In [ ]:
train(net,dataloader,val_x,val_lab)

## Network ko class ke roop mein define karna

`torch.nn.Module` se inherit hui class use karna zyada flexible tareeka hai, kyunki hum uske andar koi bhi computation define kar sakte hain. `Module` kai cheezein automate kar deta hai, jaise yeh automatically un sabhi internal variables ko pehchaan leta hai jo PyTorch layers hain, aur unke parameters ko optimization ke liye collect kar leta hai. Aapko bas network ki saari layers ko class ke members ke roop mein define karna hota hai:

In [ ]:
class MyNet(torch.nn.Module):
  def __init__(self,hidden_size=10,func=torch.nn.Sigmoid()):
    super().__init__()
    self.fc1 = torch.nn.Linear(2,hidden_size)
    self.func = func
    self.fc2 = torch.nn.Linear(hidden_size,1)

  def forward(self,x):
    x = self.fc1(x)
    x = self.func(x)
    x = self.fc2(x)
    return x
  
net = MyNet(func=torch.nn.ReLU())
print(net)

In [ ]:
train(net,dataloader,val_x,val_lab,lr=0.005)

**Task 1**: Training ke dauran training aur validation data par loss function aur accuracy ke graphs plot karein

**Task 2**: Is code ka use karke MNIST classification problem solve karne ki koshish karein. Hint: `crossentropy_with_logits` ko loss function ke roop mein use karein.

Chaliye likhe hue PyTorch model ko PyTorch Lightning module mein wrap karte hain. Isse aap apne model ke saath zyada convenient aur flexible tareeke se kaam kar sakte hain, training aur accuracy testing ke liye alag-alag Lightning methods use karke.

Sabse pehle hamein PyTorch Lightning install aur import karna hoga. Ise neeche diye gaye command se kiya ja sakta hai:

```
pip install pytorch-lightning
```
ya
```
conda install -c conda-forge pytorch-lightning
```

In [ ]:
import pytorch_lightning as pl

Lightning mein hamara code kaam kare, iske liye hamein yeh karna hoga:

1. `pl.LightningModule` ka subclass banayein aur usme model architecture ko `__init__` method aur `forward` pass method mein daalein.
2. Use kiye gaye optimizer ko `configure_optimizers()` method mein shift karein.
3. Training aur validation process ko respectively `training_step` aur `validation_step` methods mein define karein.
4. (Optional) Testing (`test_step` method) aur prediction process (`predict_step` method) implement karein.

Yeh bhi samajhna zaroori hai ki PyTorch Lightning mein models ko alag-alag devices par move karne ki built-in capability hoti hai, jo `DataLoaders` se aane wale data ki location par depend karti hai. Isliye code se saare `.cuda()` ya `.to(device)` calls hata dene chahiye.

In [ ]:
class MyNetPL(pl.LightningModule):
    def __init__(self, hidden_size = 10, func = torch.nn.Sigmoid()):
        super().__init__()
        self.fc1 = torch.nn.Linear(2,hidden_size)
        self.func = func
        self.fc2 = torch.nn.Linear(hidden_size,1)

        self.val_epoch_num = 0 # for logging

    def forward(self, x):
        x = self.fc1(x)
        x = self.func(x)
        x = self.fc2(x)
        return x

    def training_step(self, batch, batch_nb):
        x, y = batch
        y_res = self(x).view(-1)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(y_res, y)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr = 0.005)
        return optimizer
    
    def validation_step(self, batch, batch_nb):
        x, y = batch
        y_res = self(x).view(-1)
        val_loss = torch.nn.functional.binary_cross_entropy_with_logits(y_res, y)
        print("Epoch ", self.val_epoch_num, ": val loss = ", val_loss.item(), " val acc = ",((torch.sigmoid(y_res.flatten())>0.5).float()==y).float().mean().item(),  sep = "")
        self.val_epoch_num += 1

Chaliye validation `Dataset` aur `DataLoader` bhi add karte hain:

In [ ]:
valid_dataset = torch.utils.data.TensorDataset(torch.tensor(valid_x),torch.tensor(valid_labels,dtype=torch.float32))
valid_dataloader = torch.utils.data.DataLoader(valid_dataset, batch_size = 16)

Ab hamara model training ke liye ready hai. Pytorch Lightning mein, yeh process `Trainer` class ke ek object ke zariye implement hota hai, jo basically model ko kisi bhi datasets ke saath "mix" karta hai.

In [ ]:
net = MyNetPL(func=torch.nn.ReLU())
trainer = pl.Trainer(max_epochs = 30, log_every_n_steps = 1, accelerator='gpu', devices=1)
trainer.fit(model = net, train_dataloaders = dataloader, val_dataloaders = valid_dataloader)

## Main baatein

* PyTorch aapko tensors par low level par kaam karne deta hai, jisse aapko maximum flexibility milti hai.
* Data ke saath kaam karne ke liye convenient tools available hain, jaise Datasets aur Dataloaders.
* Aap `Sequential` syntax use karke ya `torch.nn.Module` se class inherit karke neural network architectures define kar sakte hain.
* Network ko define aur train karne ke aur bhi simple approach ke liye - PyTorch Lightning par dhyan dein.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->